# 03 — Query 3: Temporal Seasonality & Multi-Year Forecasting with Prophet

- **Owner:** Romain Thepaut (`@RomThpt`)
- **Course Rubric:** Query 3 `[2 pts]` (reusing Lab 7 `prophet.Prophet` time-series forecasting)

In [1]:
# Import libraries and project functions
import os
import sys
import warnings
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_shark_data, clean_shark_data
from src.indicators import query_temporal_forecast

warnings.filterwarnings('ignore')
clean_sharks = clean_shark_data(load_shark_data('../data/attacks.csv'))
print(f'Loaded {len(clean_sharks)} cleaned incidents spanning {clean_sharks["Year"].min()}-{clean_sharks["Year"].max()}.')

Loaded 4033 cleaned incidents spanning 1960-2018.


## Query 3 — Hemisphere Seasonality, Annual Forecast & Monthly Decomposition `[2 pts]`

### Indicator Explanation
- **Calculation:**
  1. Computed normalized monthly incident shares (%) separately for the **Northern Hemisphere** (`United States`) and **Southern Hemisphere** (`Australia` & `South Africa`).
  2. Aggregated complete annual counts (`ds`, `y`) from 1960 to 2017 (automatically excluding the partial 2018 cut-off year so Prophet trend estimation is not biased downward) and fit `Prophet(interval_width=0.95)` to forecast the next **5 years** (`yhat`, `yhat_lower`, `yhat_upper`).
  3. Fit a second **monthly Prophet model** (`yearly_seasonality=True`) to decompose the global series into its long-term trend and yearly seasonal cycle (Lab 7 pattern).
- **What it represents:** Quantifies how water temperature and beach tourism seasons drive monthly shark encounters, and projects expected global encounter volume with 95% confidence bounds.
- **Interpretation:**
  - Monthly peaks are **completely inverted** across hemispheres: US attacks surge in **July-September**, whereas Australia & South Africa peak in **December-March** (local summer).
  - Annual global counts grew steadily from ~40-60/year in the 1960s-1980s to ~105-120/year in the 2010s. Prophet projects a stabilized upward baseline of **112 to 118 annual incidents** over the next 5 years (95% CI roughly 67 to 163).

In [2]:
# Execute Query 3 temporal analysis and Prophet models
q3 = query_temporal_forecast(clean_sharks, forecast_years=5)

print('--- Monthly Seasonality Shares (%) ---')
display(q3['season_df'])
q3['fig_seasonality'].show()

17:44:21 - cmdstanpy - INFO - Chain [1] start processing


17:44:21 - cmdstanpy - INFO - Chain [1] done processing


17:44:21 - cmdstanpy - INFO - Chain [1] start processing


17:44:21 - cmdstanpy - INFO - Chain [1] done processing


--- Monthly Seasonality Shares (%) ---


,Month,Northern_Hemisphere_US,Southern_Hemisphere_Aus_SA
0,Jan,2.49,15.80
1,Feb,2.77,11.25
2,Mar,4.86,9.11
3,Apr,8.27,8.75
4,May,7.69,5.54
5,Jun,11.39,5.36
6,Jul,14.45,5.80
7,Aug,14.51,4.55
8,Sep,13.93,5.89
9,Oct,10.64,7.41


In [3]:
# Display 5-year Prophet forecast table and interactive charts
print('--- 5-Year Annual Prophet Forecast ---')
display(q3['forecast_df'].tail(5))
q3['fig_forecast'].show()

print('--- Monthly Prophet Trend & Yearly Seasonality Decomposition ---')
q3['fig_components'].show()

--- 5-Year Annual Prophet Forecast ---


,ds,yhat,yhat_lower,yhat_upper
58,2018-01-01,111.902353,65.521997,156.047308
59,2019-01-01,113.372402,67.966092,156.335400
60,2020-01-01,114.842452,65.532395,162.855435
61,2021-01-01,116.316529,76.086557,163.399746
62,2022-01-01,117.786578,75.914673,164.316264


--- Monthly Prophet Trend & Yearly Seasonality Decomposition ---
